# MCQ — Trả lời 10 câu hỏi trắc nghiệm bằng dữ liệu

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = 'datathon-2026-round-1/'
orders      = pd.read_csv(DATA + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA + 'order_items.csv', low_memory=False)
products    = pd.read_csv(DATA + 'products.csv')
customers   = pd.read_csv(DATA + 'customers.csv')
returns     = pd.read_csv(DATA + 'returns.csv')
payments    = pd.read_csv(DATA + 'payments.csv')
web_traffic = pd.read_csv(DATA + 'web_traffic.csv')
geography   = pd.read_csv(DATA + 'geography.csv')
sales       = pd.read_csv(DATA + 'sales.csv', parse_dates=['Date'])
print('All data loaded.')

All data loaded.


In [2]:
# ============================================================
# Q1: Trung vị inter-order gap (ngày) của khách có > 1 đơn
# ============================================================
print('=' * 70)
print('Q1: Trung vị inter-order gap')
print('=' * 70)

# Sắp xếp theo customer_id và order_date
o = orders.sort_values(['customer_id', 'order_date'])
# Tính khoảng cách giữa 2 đơn liên tiếp
o['prev_date'] = o.groupby('customer_id')['order_date'].shift(1)
o['gap_days'] = (o['order_date'] - o['prev_date']).dt.days

# Chỉ lấy khách có > 1 đơn -> gap_days not null
gaps = o.dropna(subset=['gap_days'])
median_gap = gaps['gap_days'].median()
mean_gap = gaps['gap_days'].mean()

print(f'  Số khoảng gap: {len(gaps):,}')
print(f'  Trung vị gap:  {median_gap:.1f} ngày')
print(f'  Trung bình gap: {mean_gap:.1f} ngày')
print(f'  Quartiles: {gaps["gap_days"].quantile([0.25, 0.5, 0.75]).to_dict()}')

if median_gap < 60:
    print('  → ĐÁP ÁN: A) 30 ngày')
elif median_gap < 135:
    print('  → ĐÁP ÁN: B) 90 ngày')
elif median_gap < 270:
    print('  → ĐÁP ÁN: C) 144 ngày')
else:
    print('  → ĐÁP ÁN: D) 365 ngày')

Q1: Trung vị inter-order gap


  Số khoảng gap: 556,699
  Trung vị gap:  144.0 ngày
  Trung bình gap: 285.6 ngày
  Quartiles: {0.25: 46.0, 0.5: 144.0, 0.75: 357.0}
  → ĐÁP ÁN: C) 144 ngày


In [3]:
# ============================================================
# Q2: Segment có gross margin trung bình cao nhất
# ============================================================
print('=' * 70)
print('Q2: Segment có (price - cogs) / price cao nhất')
print('=' * 70)

products['gross_margin_pct'] = (products['price'] - products['cogs']) / products['price']
margin_by_seg = products.groupby('segment')['gross_margin_pct'].mean().sort_values(ascending=False)
print(margin_by_seg.to_string())
print(f'\n  → ĐÁP ÁN: {margin_by_seg.idxmax()}')

Q2: Segment có (price - cogs) / price cao nhất
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343

  → ĐÁP ÁN: Standard


In [4]:
# ============================================================
# Q3: Lý do trả hàng phổ biến nhất cho Streetwear
# ============================================================
print('=' * 70)
print('Q3: Lý do trả hàng nhiều nhất — Streetwear')
print('=' * 70)

# returns có product_id không? Kiểm tra
print(f'  returns columns: {returns.columns.tolist()}')

# Join returns -> order_items (product_id) -> products (category)
if 'product_id' in returns.columns:
    ret_prod = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
else:
    # returns uses order_id -> join to order_items first
    ret_oi = returns.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
    ret_prod = ret_oi.merge(products[['product_id', 'category']], on='product_id', how='left')

streetwear_returns = ret_prod[ret_prod['category'] == 'Streetwear']
reason_counts = streetwear_returns['return_reason'].value_counts()
print(reason_counts.to_string())
print(f'\n  → ĐÁP ÁN: {reason_counts.idxmax()}')

Q3: Lý do trả hàng nhiều nhất — Streetwear
  returns columns: ['return_id', 'order_id', 'product_id', 'return_date', 'return_reason', 'return_quantity', 'refund_amount']
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159

  → ĐÁP ÁN: wrong_size


In [5]:
# ============================================================
# Q4: Traffic source có bounce_rate trung bình thấp nhất
# ============================================================
print('=' * 70)
print('Q4: Traffic source — bounce_rate thấp nhất')
print('=' * 70)

bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(bounce_by_source.to_string())
print(f'\n  → ĐÁP ÁN: {bounce_by_source.idxmin()}')

Q4: Traffic source — bounce_rate thấp nhất
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511

  → ĐÁP ÁN: email_campaign


In [6]:
# ============================================================
# Q5: % dòng order_items có promo_id không null
# ============================================================
print('=' * 70)
print('Q5: % order_items có promo_id')
print('=' * 70)

has_promo = order_items['promo_id'].notna().mean() * 100
print(f'  % có promo_id:  {has_promo:.1f}%')
print(f'  % có promo_id_2: {order_items["promo_id_2"].notna().mean()*100:.1f}%')

# Câu hỏi nói "promo_id không null"
if has_promo < 18:
    print('  → ĐÁP ÁN: A) 12%')
elif has_promo < 32:
    print('  → ĐÁP ÁN: B) 25%')
elif has_promo < 46:
    print('  → ĐÁP ÁN: C) 39%')
else:
    print('  → ĐÁP ÁN: D) 54%')

Q5: % order_items có promo_id
  % có promo_id:  38.7%
  % có promo_id_2: 0.0%
  → ĐÁP ÁN: C) 39%


In [7]:
# ============================================================
# Q6: Age group có số đơn TB/khách cao nhất
# ============================================================
print('=' * 70)
print('Q6: Age group — đơn TB/khách cao nhất')
print('=' * 70)

# Join orders -> customers
oc = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')
oc = oc[oc['age_group'].notna()]

# Đếm số đơn và số khách theo age_group
agg = oc.groupby('age_group').agg(
    total_orders=('order_id', 'count'),
    total_customers=('customer_id', 'nunique')
)
agg['avg_orders_per_customer'] = agg['total_orders'] / agg['total_customers']
agg = agg.sort_values('avg_orders_per_customer', ascending=False)
print(agg.to_string())
print(f'\n  → ĐÁP ÁN: {agg["avg_orders_per_customer"].idxmax()}')

Q6: Age group — đơn TB/khách cao nhất


           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            10010                 7.268731
45-54            124138            17193                 7.220264
35-44            170368            23642                 7.206159
25-34            190622            26802                 7.112230
18-24             89057            12599                 7.068577

  → ĐÁP ÁN: 55+


In [8]:
# ============================================================
# Q7: Region nào tạo tổng Revenue cao nhất
# ============================================================
print('=' * 70)
print('Q7: Region — tổng Revenue cao nhất (từ sales/orders + geography)')
print('=' * 70)

# sales.csv chỉ có Date, Revenue, COGS -> không có customer/zip
# Phải dùng orders -> customers -> geography -> order_items
oi = order_items.copy()
oi['line_total'] = oi['quantity'] * oi['unit_price']

oi_orders = oi.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')
oi_cust = oi_orders.merge(customers[['customer_id', 'zip']], on='customer_id', how='left')
oi_geo = oi_cust.merge(geography[['zip', 'region']], on='zip', how='left')

rev_by_region = oi_geo.groupby('region')['line_total'].sum().sort_values(ascending=False)
print(rev_by_region.to_string())
print(f'\n  Max: {rev_by_region.idxmax()} = {rev_by_region.max():,.0f}')
print(f'  Min: {rev_by_region.idxmin()} = {rev_by_region.min():,.0f}')
ratio = rev_by_region.max() / rev_by_region.min()
print(f'  Max/Min ratio: {ratio:.2f}')

if ratio < 1.1:
    print('  → ĐÁP ÁN: D) Cả ba vùng xấp xỉ bằng nhau')
else:
    print(f'  → ĐÁP ÁN: {rev_by_region.idxmax()}')

Q7: Region — tổng Revenue cao nhất (từ sales/orders + geography)


region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09

  Max: East = 7,637,532,676
  Min: West = 3,851,035,438
  Max/Min ratio: 1.98
  → ĐÁP ÁN: East


In [9]:
# ============================================================
# Q8: Phương thức thanh toán phổ biến nhất cho đơn cancelled
# ============================================================
print('=' * 70)
print('Q8: Payment method phổ biến nhất — đơn cancelled')
print('=' * 70)

cancelled_ids = orders[orders['order_status'] == 'cancelled']['order_id']
cancelled_payments = payments[payments['order_id'].isin(cancelled_ids)]
pm_counts = cancelled_payments['payment_method'].value_counts()
print(pm_counts.to_string())
print(f'\n  → ĐÁP ÁN: {pm_counts.idxmax()}')

Q8: Payment method phổ biến nhất — đơn cancelled
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

  → ĐÁP ÁN: credit_card


In [10]:
# ============================================================
# Q9: Size nào có tỷ lệ trả hàng cao nhất
# ============================================================
print('=' * 70)
print('Q9: Size — tỷ lệ trả hàng cao nhất')
print('=' * 70)

# order_items + products -> count by size
oi_prod = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_by_size = oi_prod.groupby('size')['order_id'].count().rename('n_oi')

# returns + order_items + products -> count by size
if 'product_id' in returns.columns:
    ret_prod = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
else:
    ret_oi = returns.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
    ret_prod = ret_oi.merge(products[['product_id', 'size']], on='product_id', how='left')

ret_by_size = ret_prod.groupby('size')['order_id'].count().rename('n_returns')

size_rates = pd.DataFrame({'n_oi': oi_by_size, 'n_returns': ret_by_size}).dropna()
size_rates['return_rate'] = size_rates['n_returns'] / size_rates['n_oi']
size_rates = size_rates.sort_values('return_rate', ascending=False)
print(size_rates.to_string())
print(f'\n  → ĐÁP ÁN: {size_rates["return_rate"].idxmax()}')

Q9: Size — tỷ lệ trả hàng cao nhất
        n_oi  n_returns  return_rate
size                                
S     172042       9723     0.056515
L     173174       9741     0.056250
M     176428       9820     0.055660
XL    193025      10655     0.055200

  → ĐÁP ÁN: S


In [11]:
# ============================================================
# Q10: Installment plan nào có payment_value TB/đơn cao nhất
# ============================================================
print('=' * 70)
print('Q10: Installment plan — payment_value TB cao nhất')
print('=' * 70)

avg_pv = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(avg_pv.to_string())
print(f'\n  → ĐÁP ÁN: {avg_pv.idxmax()} kỳ')

Q10: Installment plan — payment_value TB cao nhất
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729

  → ĐÁP ÁN: 6 kỳ
